<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10, Part 2 discussion — does kernel size matter, on the real localization data?

The main notebook's 1D-CNN used a fixed kernel size of 7. Here: re-fetch the exact same subcellular-localization dataset and homology-aware split from Days 8-10 (MMseqs2 clusters at 30% identity, whole clusters per split), and compare three kernel sizes (3, 7, 15) on real receptive field and real measured validation accuracy, rather than assuming 7 was the right choice.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

import re
import requests

UNIPROT_CLASSES = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
}
CLASS_NAMES = list(UNIPROT_CLASSES.keys())

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                "format": "tsv", "size": size},
        timeout=60,
    )
    r.raise_for_status()
    rows = []
    for line in r.text.strip().split("\n")[1:]:
        parts = line.split("\t")
        if len(parts) == 3:
            rows.append(tuple(parts))
    return rows

def is_unambiguous_single_location(location_text, target):
    location_text = re.sub(r"Note=.*", "", location_text)
    location_text = re.sub(r"\{[^}]*\}", "", location_text)
    if "Isoform" in location_text:
        return False
    terms = set()
    for statement in [s.strip() for s in location_text.split(".") if s.strip()]:
        body = statement.split(":", 1)[-1] if ":" in statement else statement
        top_term = re.split(r"[,;]", body)[0].strip().rstrip(".")
        if top_term:
            terms.add(top_term)
    return terms == {target}

entries_by_class = {name: [] for name in CLASS_NAMES}
for name, code in UNIPROT_CLASSES.items():
    rows = fetch_uniprot(code, size=500)
    kept = [(acc, seq) for (acc, seq, loc) in rows if is_unambiguous_single_location(loc, name)]
    entries_by_class[name] = kept
    print(f"{name}: fetched {len(rows)}, unambiguous single-location {len(kept)}")

Cytoplasm: fetched 500, unambiguous single-location 424


Nucleus: fetched 500, unambiguous single-location 497


Mitochondrion: fetched 500, unambiguous single-location 140


Secreted: fetched 500, unambiguous single-location 488


Cell membrane: fetched 500, unambiguous single-location 338


In [2]:
N_PER_CLASS = min(len(v) for v in entries_by_class.values())
print("balancing every class to", N_PER_CLASS, "sequences")

rng = np.random.RandomState(0)
balanced_accs, balanced_seqs, balanced_labels = [], [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    pool = entries_by_class[class_name]
    for i in rng.choice(len(pool), N_PER_CLASS, replace=False):
        balanced_accs.append(pool[i][0])
        balanced_seqs.append(pool[i][1])
        balanced_labels.append(class_idx)

print("total balanced dataset:", len(balanced_seqs), "sequences,", len(CLASS_NAMES), "classes")
print("(Day 8/9 reported 700 sequences, 140/class -- checking this matches)")

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
SEQ_LEN = 150

def one_hot_encode(sequence, length=SEQ_LEN):
    encoded = np.zeros((length, len(AMINO_ACIDS)), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        if residue in AA_TO_INDEX:
            encoded[position, AA_TO_INDEX[residue]] = 1.0
    return encoded  # (SEQ_LEN, 20) -- NOT flattened this time, a CNN wants the 2D shape

X = np.stack([one_hot_encode(s) for s in balanced_seqs])       # (N, 150, 20)
X = X.transpose(0, 2, 1)                                        # (N, 20, 150): channels=amino acids, length=position
y = np.array(balanced_labels)
print("X shape:", X.shape, " (N, channels=20 amino acids, length=150 positions)")
print("y shape:", y.shape)

balancing every class to 140 sequences
total balanced dataset: 700 sequences, 5 classes
(Day 8/9 reported 700 sequences, 140/class -- checking this matches)
X shape: (700, 20, 150)  (N, channels=20 amino acids, length=150 positions)
y shape: (700,)


In [3]:
import os, shutil, subprocess, tempfile

# Day 8's homology-aware split: MMseqs2 clusters at 30% identity, whole clusters per split
CLUSTER_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/day08-subcell-mmseqs-30.tsv"
LOCAL_TSV = "data/day08-subcell-mmseqs-30.tsv"

def mmseqs_clusters(accs, seqs, min_seq_id=0.3):
    tmp = tempfile.mkdtemp()
    with open(f"{tmp}/in.fasta", "w") as f:
        for a, s in zip(accs, seqs):
            f.write(f">{a}\n{s}\n")
    subprocess.run(["mmseqs", "easy-cluster", f"{tmp}/in.fasta", f"{tmp}/clu", f"{tmp}/work",
                    "--min-seq-id", str(min_seq_id), "-c", "0.5", "-v", "1"],
                   check=True, stdout=subprocess.DEVNULL)
    pairs = [line.split("\t") for line in open(f"{tmp}/clu_cluster.tsv").read().split("\n") if line]
    shutil.rmtree(tmp)
    return {member: rep for rep, member in pairs}

def load_precomputed_clusters():
    text = open(LOCAL_TSV).read() if os.path.exists(LOCAL_TSV) else requests.get(CLUSTER_URL, timeout=30).text
    pairs = [line.split("\t") for line in text.strip().split("\n")[1:]]
    return {member: rep for rep, member in pairs}

if shutil.which("mmseqs"):
    member_to_rep = mmseqs_clusters(balanced_accs, balanced_seqs)
    print("clustered with local MMseqs2 (30% identity, 50% coverage)")
else:
    member_to_rep = load_precomputed_clusters()
    print("mmseqs not found -- loaded the precomputed MMseqs2 clustering")

# proteins missing from the clustering (e.g. if UniProt changed) become their own cluster
reps = [member_to_rep.get(a, a) for a in balanced_accs]
rep_ids = {r: i for i, r in enumerate(sorted(set(reps)))}
groups = np.array([rep_ids[r] for r in reps])
sizes = np.bincount(groups)
print(f"{len(y)} proteins -> {len(sizes)} clusters; "
      f"{(sizes > 1).sum()} clusters have more than one member ({sizes[sizes > 1].sum()} proteins), largest has {sizes.max()}")

def homology_split(y, groups, fractions=(0.70, 0.15, 0.15), seed=0):
    '''Assign whole clusters, in random order, to train/val/test: each cluster
    goes to the split furthest below its target share of that cluster's
    classes (ties broken at random). Returns three index arrays.'''
    rng = np.random.RandomState(seed)
    n_classes = y.max() + 1
    target = np.outer(fractions, np.bincount(y, minlength=n_classes))   # (3 splits, n_classes)
    have = np.zeros_like(target)
    assignment = {}
    for g in rng.permutation(np.unique(groups)):
        cls_counts = np.bincount(y[groups == g], minlength=n_classes)
        deficit = ((target - have) * (cls_counts > 0)).sum(axis=1) / target.sum(axis=1)
        best = np.flatnonzero(deficit == deficit.max())
        split = int(rng.choice(best))
        assignment[g] = split
        have[split] += cls_counts
    split_of = np.array([assignment[g] for g in groups])
    return [np.where(split_of == k)[0] for k in range(3)]

tr_h, va_h, te_h = homology_split(y, groups)
X_train, y_train = X[tr_h], y[tr_h]
X_val, y_val = X[va_h], y[va_h]
X_test, y_test = X[te_h], y[te_h]
print(f"train: {len(X_train)}   val: {len(X_val)}   test: {len(X_test)}")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

clustered with local MMseqs2 (30% identity, 50% coverage)
700 proteins -> 551 clusters; 75 clusters have more than one member (224 proteins), largest has 11
train: 486   val: 105   test: 109


## Three kernel sizes, same architecture otherwise

Each filter's **receptive field** after two stacked convolutional layers is
$k + (k - 1) = 2k - 1$ residues (each layer adds $k-1$ to how far a single output position can "see" back along the sequence). Train the same two-conv-layer architecture with kernel size 3, 7, and 15, and compare real measured validation accuracy and parameter count.

In [4]:
class KernelSizeCNN(nn.Module):
    def __init__(self, kernel_size, n_classes=5, n_channels=20):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=kernel_size, padding=pad)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=kernel_size, padding=pad)
        self.pool = nn.MaxPool1d(2)
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = torch.adaptive_avg_pool1d(x, 1).squeeze(-1)
        return self.classifier(x)


def train_and_eval(kernel_size, epochs=60, lr=1e-3):
    torch.manual_seed(0)
    model = KernelSizeCNN(kernel_size)
    n_params = sum(p.numel() for p in model.parameters())
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = (model(X_val_t).argmax(1) == y_val_t).float().mean().item()
        best_val_acc = max(best_val_acc, val_acc)
    receptive_field = 2 * kernel_size - 1
    return n_params, best_val_acc, receptive_field


print(f"{'kernel':>6} {'receptive field':>16} {'params':>10} {'best val acc':>13}")
results = {}
for k in (3, 7, 15):
    n_params, best_val_acc, rf = train_and_eval(k)
    results[k] = (n_params, best_val_acc, rf)
    print(f"{k:>6} {rf:>16} {n_params:>10,} {best_val_acc:>13.3f}")

kernel  receptive field     params  best val acc


     3                5      8,485         0.505


     7               13     19,237         0.581


    15               29     40,741         0.619


## Discuss

1. Before running the cell above, guess: will the largest kernel (15) win? A bigger kernel sees more context per layer, but also has more weights to learn from only 486 training examples.
2. On this homology-aware split, the largest kernel came out ahead (0.619 vs. 0.581 for kernel 7 and 0.505 for kernel 3). What biological signals might need a receptive field of ~29 residues? (Think of signal peptides and transmembrane helices.)
3. "Best val acc" here is the maximum over 60 epochs, measured on the same set used to pick it. Using Day 8's reasoning, why is this optimistic, and are differences of a few points on 105 proteins convincing?

*One group presents.*